In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Article-Level Political Bias Classification

This notebook evaluates the performance of multiple feature engineering strategies for classifying political bias at the article level.

Key steps:
- Load and preprocess dataset
- Apply feature extraction methods
- Perform stratified cross-validation
- Evaluate models (Logistic Regression, SVC)
- Analyze results

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import load_dataset

# Project modules
from src.preprocessing import load_external_dataframe
from src.features import (
    build_tfidf,
    build_tfidf_nmf,
    build_full,
    build_embeddings,
    build_chunked
)
from src.experiments import run_experiments
from src.utils import results_to_df

In [5]:
# Load AG News
dataset = load_dataset("ag_news")
df_raw = dataset["train"].to_pandas()

# Convert to binary
df_raw["label"] = df_raw["label"].apply(lambda x: 0 if x < 2 else 1)

# Use your abstraction
df = load_external_dataframe(
    df_raw,
    text_col="text",
    label_col="label",
    source_name="AGNews"
)

df.head()

,text,label,source
0,Wall St. Bears Claw Back Into the Black (Reute...,1,AGNews
1,Carlyle Looks Toward Commercial Aerospace (Reu...,1,AGNews
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,1,AGNews
3,Iraq Halts Oil Exports from Main Southern Pipe...,1,AGNews
4,"Oil prices soar to all-time record, posing new...",1,AGNews


In [6]:
print("Shape:", df.shape)
print("\nClass Distribution:")
print(df["label"].value_counts(normalize=True))

Shape: (120000, 3)

Class Distribution:
label
1    0.5
0    0.5
Name: proportion, dtype: float64


In [7]:
texts = df["text"].values
y = df["label"].values
groups = df["source"].values  # kept for consistency

from sklearn.model_selection import train_test_split

texts_train, texts_test, y_train, y_test, _, _ = train_test_split(
    texts,
    y,
    groups,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(texts_train))
print("Test size:", len(texts_test))

Train size: 96000
Test size: 24000


In [8]:
experiments = [
    ("TF-IDF", build_tfidf),
    ("TF-IDF + NMF", build_tfidf_nmf),
    ("Full Linguistic", build_full),
    ("Embeddings", build_embeddings),
    ("Chunked Embeddings", build_chunked),
]

In [9]:
cv_folds, cv_summary, test_res, cms = run_experiments(
    experiments,
    texts_train,
    y_train,
    texts_test,
    y_test,
    grouped=False  # IMPORTANT: article-level
)


RUNNING: TF-IDF

===== CV: TF-IDF =====
Fold 1: LR=0.959, SVC=0.961
Fold 2: LR=0.960, SVC=0.959
Fold 3: LR=0.958, SVC=0.959
Fold 4: LR=0.959, SVC=0.960
Fold 5: LR=0.959, SVC=0.961

===== FINAL TEST: TF-IDF =====

LR:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96     12000
           1       0.96      0.96      0.96     12000

    accuracy                           0.96     24000
   macro avg       0.96      0.96      0.96     24000
weighted avg       0.96      0.96      0.96     24000


SVC:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96     12000
           1       0.96      0.96      0.96     12000

    accuracy                           0.96     24000
   macro avg       0.96      0.96      0.96     24000
weighted avg       0.96      0.96      0.96     24000


RUNNING: TF-IDF + NMF

===== CV: TF-IDF + NMF =====
Fold 1: LR=0.959, SVC=0.961
Fold 2: LR=0.960, SVC=0.959
Fold 3: LR=0.

KeyboardInterrupt: 

In [ ]:
df_folds, df_summary, df_test = results_to_df(
    cv_folds,
    cv_summary,
    test_res
)

df_summary

In [ ]:
df_test

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

x = range(len(df_summary))

plt.bar(x, df_summary["SVC Mean"], label="SVC")
plt.bar(x, df_summary["LR Mean"], alpha=0.7, label="Logistic Regression")

plt.xticks(x, df_summary["Model"], rotation=30)
plt.ylabel("Accuracy")
plt.title("Cross-Validation Performance (Article-Level)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_model = df_test.sort_values(by="SVC Test Acc", ascending=False).iloc[0]["Model"]

cm = cms[best_model]["TEST_SVC"]

ConfusionMatrixDisplay(cm).plot()
plt.title(f"{best_model} - SVC (Test)")
plt.show()

## Observations

- TF-IDF-based models achieve very high performance (~0.95+ accuracy), indicating strong lexical separability between classes.
- SVC consistently outperforms Logistic Regression across all feature combinations.
- Adding linguistic features (POS, sentiment, readability) provides limited improvement.
- Sentence embeddings underperform compared to TF-IDF, suggesting that keyword-level distinctions dominate in this dataset.
- Chunked embeddings slightly improve over standard embeddings, indicating that longer context can be useful.

This confirms that topic-based classification tasks are significantly easier than political bias detection, as they rely on distinct vocabularies rather than subtle linguistic cues.

In [ ]:
df_summary.to_csv("../outputs/tables/article_cv_summary.csv", index=False)
df_test.to_csv("../outputs/tables/article_test_results.csv", index=False)

print("Saved results to outputs/tables/")